### Install Dependencies


In [1]:
pip install pillow opencv-python torchvision torch scikit-learn pandas numpy tqdm


  Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl.metadata (60 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
     ---------------------------------------- 0.0/60.8 kB ? eta -:--:--
     --------------------------------- ------ 51.2/60.8 kB 1.3 MB/s eta 0:00:01
     --------------------------------- ------ 51.2/60.8 kB 1.3 MB/s eta 0:00:01
     --------------------------------- ------ 51.2/60.8 kB 1.3 MB/s eta 0:00:01
     -------------------------------------- 60.8/60.8 kB 358.3 kB/s eta 0:00:00
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/7.0 MB ? eta -:--:--
    --------------------------------------- 0.1/7.0 MB 2.4 MB/s eta 0:00:03
   - -------------------------------------- 0.3/7.0 MB 2.9 MB/s eta 0:00:03
   -- ------------------------------------- 0.5/7.0 MB 3.4 MB/s eta 0:00:02
   --- ------------------------------------ 0.


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ----------------------------------- ---- 190.8/216.1 MB 3.6 MB/s eta 0:00:07
   ----------------------------------- ---- 191.0/216.1 MB 3.6 MB/s eta 0:00:07
   ----------------------------------- ---- 191.2/216.1 MB 3.7 MB/s eta 0:00:07
   ----------------------------------- ---- 191.3/216.1 MB 3.6 MB/s eta 0:00:07
   ----------------------------------- ---- 191.5/216.1 MB 3.6 MB/s eta 0:00:07
   ----------------------------------- ---- 191.7/216.1 MB 3.6 MB/s eta 0:00:07
   ----------------------------------- ---- 191.9/216.1 MB 3.6 MB/s eta 0:00:07
   ----------------------------------- ---- 192.1/216.1 MB 3.7 MB/s eta 0:00:07
   ----------------------------------- ---- 192.4/216.1 MB 3.8 MB/s eta 0:00:07
   ----------------------------------- ---- 192.5/216.1 MB 3.7 MB/s eta 0:00:07
   ----------------------------------- ---- 192.8/216.1 MB 3.9 MB/s eta 0:00:07
   ----------------------------------- ---- 193.0/216.1 MB 3.8 MB/s eta 0:00:07
   ----------------------------------- -

### Step 1: Import Libraries and Configure Paths

In [6]:
import json
import os
import random
from collections import defaultdict, Counter
import cv2
import numpy as np
from PIL import Image
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import shutil
import torchvision.transforms as transforms
import tarfile

In [3]:
# Configuration
DATASET_DIR = r"C:\Users\Meet\Desktop\Yelp Photos"  # Use raw string to handle Windows paths
TAR_FILE = os.path.join(DATASET_DIR, "yelp_photos.tar")
EXTRACTED_DIR = os.path.join(DATASET_DIR, "yelp_photos_extracted")  # Extracted contents
PHOTO_DIR = os.path.join(EXTRACTED_DIR, "photos")  # Directory with .jpg files
METADATA_PATH = os.path.join(EXTRACTED_DIR, "photos.json")  # Metadata file
OUTPUT_DIR = os.path.join(DATASET_DIR, "final_processed_data")  # Output directory
TARGET_SIZE = (224, 224)  # Standard size for CNNs
SAMPLE_FRACTION = 0.1  # 10% stratified sampling
BATCH_SIZE = 1000  # Process images in batches to manage memory

In [7]:
# Extract tar file if not already extracted
if not os.path.exists(EXTRACTED_DIR):
    print(f"Extracting {TAR_FILE} to {EXTRACTED_DIR}...")
    with tarfile.open(TAR_FILE, "r") as tar:
        tar.extractall(path=EXTRACTED_DIR)
    print("Extraction complete!")
else:
    print(f"{EXTRACTED_DIR} already exists, skipping extraction.")

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

Extracting C:\Users\Meet\Desktop\Yelp Photos\yelp_photos.tar to C:\Users\Meet\Desktop\Yelp Photos\yelp_photos_extracted...
Extraction complete!


### Step 2: Load and Filter Metadata

In [8]:
def load_and_filter_metadata():
    """Load photos.json and filter valid images."""
    try:
        with open(METADATA_PATH, 'r') as f:
            data = [json.loads(line) for line in f]
        print(f"Total photos in metadata: {len(data)}")
        
        valid_data = []
        for item in tqdm(data, desc="Filtering metadata"):
            photo_path = os.path.join(PHOTO_DIR, f"{item['photo_id']}.jpg")
            if os.path.exists(photo_path):
                try:
                    img = Image.open(photo_path)
                    img.verify()  # Check for corruption
                    valid_data.append(item)
                except Exception as e:
                    print(f"Corrupted image: {item['photo_id']}, error: {e}")
            else:
                print(f"Missing image: {item['photo_id']}")
        
        print(f"Valid photos after filtering: {len(valid_data)}")
        
        # Save filtered metadata
        with open(os.path.join(OUTPUT_DIR, "filtered_photos.json"), 'w') as f:
            json.dump(valid_data, f, indent=2)
        
        return valid_data
    except Exception as e:
        print(f"Error loading metadata: {e}")
        return []

# Run step
photos_metadata = load_and_filter_metadata()

Total photos in metadata: 200100


Filtering metadata:   1%|          | 1606/200100 [00:37<2:04:08, 26.65it/s]

Corrupted image: ydm3g1wUWSxJnMPgHk2JhQ, error: cannot identify image file 'C:\\Users\\Meet\\Desktop\\Yelp Photos\\yelp_photos_extracted\\photos\\ydm3g1wUWSxJnMPgHk2JhQ.jpg'


Filtering metadata:   2%|▏         | 4234/200100 [01:33<1:24:40, 38.55it/s]

Corrupted image: JGpfPj8VEvnq1B-Xqr3w-A, error: cannot identify image file 'C:\\Users\\Meet\\Desktop\\Yelp Photos\\yelp_photos_extracted\\photos\\JGpfPj8VEvnq1B-Xqr3w-A.jpg'


...

Corrupted image: iX-8Xm2G7meRHUg8qhoL1A, error: cannot identify image file 'C:\\Users\\Meet\\Desktop\\Yelp Photos\\yelp_photos_extracted\\photos\\iX-8Xm2G7meRHUg8qhoL1A.jpg'


Filtering metadata: 100%|█████████▉| 199228/200100 [1:13:55<00:23, 37.89it/s]

Corrupted image: GWLmPwKeBnh2b_7Kv_LQ7w, error: cannot identify image file 'C:\\Users\\Meet\\Desktop\\Yelp Photos\\yelp_photos_extracted\\photos\\GWLmPwKeBnh2b_7Kv_LQ7w.jpg'


Filtering metadata: 100%|██████████| 200100/200100 [1:14:13<00:00, 44.93it/s]


Valid photos after filtering: 199994


### Step 3: Stratified Sampling

In [9]:
def stratified_sampling(photos_metadata, sample_fraction=0.1):
    """Perform stratified sampling to select a fraction of data per class."""
    label_to_items = defaultdict(list)
    for item in photos_metadata:
        label_to_items[item['label']].append(item)
    
    original_counts = {label: len(items) for label, items in label_to_items.items()}
    print("Original label distribution:", original_counts)
    
    sampled_data = []
    for label, items in label_to_items.items():
        k = max(1, int(len(items) * sample_fraction))
        sampled = random.sample(items, k=k)
        sampled_data.extend(sampled)
        print(f"{label}: sampled {k} out of {len(items)}")
    
    random.shuffle(sampled_data)
    print(f"Total images after stratified sampling: {len(sampled_data)}")
    
    with open(os.path.join(OUTPUT_DIR, "sampled_photos.json"), 'w') as f:
        json.dump(sampled_data, f, indent=2)
    
    return sampled_data

# Run step
sampled_metadata = stratified_sampling(photos_metadata, SAMPLE_FRACTION)

Original label distribution: {'inside': 56030, 'outside': 18569, 'drink': 15670, 'food': 108047, 'menu': 1678}
inside: sampled 5603 out of 56030
outside: sampled 1856 out of 18569
drink: sampled 1567 out of 15670
food: sampled 10804 out of 108047
menu: sampled 167 out of 1678
Total images after stratified sampling: 19997


### Step 4: Handle Data Imbalance with Augmentation

In [10]:
def balance_classes(sampled_metadata, target_count=None):
    """Balance classes by augmenting underrepresented ones."""
    label_counts = Counter(item['label'] for item in sampled_metadata)
    print("Sampled label distribution:", dict(label_counts))
    
    if target_count is None:
        target_count = int(max(label_counts.values()) * 0.25)
    print(f"Target count per class: {target_count}")
    
    augment_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(20),
        transforms.RandomResizedCrop(TARGET_SIZE, scale=(0.8, 1.0)),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2)
    ])
    
    balanced_data = []
    balanced_dir = os.path.join(OUTPUT_DIR, "balanced")
    os.makedirs(balanced_dir, exist_ok=True)
    
    classes_to_augment = {'outside', 'menu', 'drink'}
    
    for label in label_counts:
        label_dir = os.path.join(balanced_dir, label)
        os.makedirs(label_dir, exist_ok=True)
        
        label_items = [item for item in sampled_metadata if item['label'] == label]
        current_count = len(label_items)
        
        for item in label_items:
            src = os.path.join(PHOTO_DIR, f"{item['photo_id']}.jpg")
            dst = os.path.join(label_dir, f"{item['photo_id']}.jpg")
            if os.path.exists(src):
                shutil.copyfile(src, dst)
                balanced_data.append(item)
            else:
                print(f"Missing image: {src}")
        
        if label in classes_to_augment and current_count < target_count:
            print(f"Balancing {label}: Current={current_count}, Target={target_count}")
            copies_needed = target_count - current_count
            augment_per_image = max(1, copies_needed // current_count)
            extra_needed = copies_needed - (augment_per_image * current_count)
            
            for idx, item in enumerate(tqdm(label_items, desc=f"Augmenting {label}")):
                src = os.path.join(PHOTO_DIR, f"{item['photo_id']}.jpg")
                if not os.path.exists(src):
                    continue
                img = Image.open(src).convert('RGB')
                
                for i in range(augment_per_image):
                    aug_img = augment_transform(img)
                    aug_id = f"{item['photo_id']}_aug{i}"
                    aug_path = os.path.join(label_dir, f"{aug_id}.jpg")
                    aug_img.save(aug_path, quality=95)
                    balanced_data.append({
                        'photo_id': aug_id,
                        'label': label,
                        'business_id': item['business_id'],
                        'caption': item['caption']
                    })
                
                if idx < extra_needed:
                    aug_img = augment_transform(img)
                    aug_id = f"{item['photo_id']}_extra_aug"
                    aug_path = os.path.join(label_dir, f"{aug_id}.jpg")
                    aug_img.save(aug_path, quality=95)
                    balanced_data.append({
                        'photo_id': aug_id,
                        'label': label,
                        'business_id': item['business_id'],
                        'caption': item['caption']
                    })
    
    with open(os.path.join(OUTPUT_DIR, "balanced_photos.json"), 'w') as f:
        json.dump(balanced_data, f, indent=2)
    
    final_counts = Counter(item['label'] for item in balanced_data)
    print("Final balanced distribution:", dict(final_counts))
    
    return balanced_data, balanced_dir

# Run step
balanced_metadata, balanced_dir = balance_classes(sampled_metadata)

Sampled label distribution: {'food': 10804, 'drink': 1567, 'inside': 5603, 'outside': 1856, 'menu': 167}
Target count per class: 2701
Balancing drink: Current=1567, Target=2701


Augmenting drink: 100%|██████████| 1567/1567 [00:39<00:00, 39.84it/s]


Balancing outside: Current=1856, Target=2701


Augmenting outside: 100%|██████████| 1856/1856 [00:38<00:00, 48.12it/s]


Balancing menu: Current=167, Target=2701


Augmenting menu: 100%|██████████| 167/167 [00:33<00:00,  5.03it/s]


Final balanced distribution: {'food': 10804, 'drink': 3134, 'inside': 5603, 'outside': 3712, 'menu': 2701}


### Step 5: Preprocess Images

In [11]:
def preprocess_images(balanced_metadata, balanced_dir):
    """Preprocess images (resize, normalize) and save to final structure."""
    processed_images = {}
    failed_images = []
    
    def preprocess_image(image_path):
        try:
            img = cv2.imread(image_path)
            if img is None:
                img = np.array(Image.open(image_path).convert('RGB'))
                img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
            h, w = img.shape[:2]
            scale = min(TARGET_SIZE[0]/h, TARGET_SIZE[1]/w)
            new_h, new_w = int(h * scale), int(w * scale)
            img = cv2.resize(img, (new_w, new_h))
            if new_h != TARGET_SIZE[0] or new_w != TARGET_SIZE[1]:
                delta_h = TARGET_SIZE[0] - new_h
                delta_w = TARGET_SIZE[1] - new_w
                top, bottom = delta_h//2, delta_h-(delta_h//2)
                left, right = delta_w//2, delta_w-(delta_w//2)
                img = cv2.copyMakeBorder(img, top, bottom, left, right, 
                                       cv2.BORDER_CONSTANT, value=[0, 0, 0])
            return img.astype(np.float32) / 255.0
        except Exception as e:
            print(f"Error processing {image_path}: {e}")
            return None
    
    print(f"Processing images from {balanced_dir}...")
    for item in tqdm(balanced_metadata, desc="Processing images"):
        photo_id = item['photo_id']
        label = item['label']
        image_path = os.path.join(balanced_dir, label, f"{photo_id}.jpg")
        
        if not os.path.exists(image_path):
            failed_images.append(photo_id)
            continue
        
        processed_img = preprocess_image(image_path)
        if processed_img is not None:
            label_dir = os.path.join(OUTPUT_DIR, label)
            os.makedirs(label_dir, exist_ok=True)
            output_path = os.path.join(label_dir, f"{photo_id}.jpg")
            cv2.imwrite(
                output_path,
                cv2.cvtColor((processed_img * 255).astype(np.uint8), cv2.COLOR_RGB2BGR),
                [int(cv2.IMWRITE_JPEG_QUALITY), 95]
            )
            processed_images[photo_id] = {
                'path': output_path,
                'label': label,
                'business_id': item['business_id'],
                'caption': item['caption']
            }
    
    with open(os.path.join(OUTPUT_DIR, "processing_log.json"), 'w') as f:
        json.dump({
            'processed': len(processed_images),
            'failed': len(failed_images),
            'failed_samples': failed_images,
            'class_distribution': {
                label: len([v for v in processed_images.values() if v['label'] == label])
                for label in set(item['label'] for item in balanced_metadata)
            }
        }, f, indent=2)
    
    print(f"Successfully processed: {len(processed_images)} images")
    print(f"Failed to process: {len(failed_images)} images")
    if failed_images:
        print("First 10 failed images:", failed_images[:10])
    
    return processed_images

# Run step
processed_images = preprocess_images(balanced_metadata, balanced_dir)

Processing images from C:\Users\Meet\Desktop\Yelp Photos\final_processed_data\balanced...


Processing images: 100%|██████████| 25954/25954 [14:49<00:00, 29.16it/s]


Successfully processed: 25954 images
Failed to process: 0 images


### Step 6: Split Data with Leakage Prevention

In [12]:
def create_train_val_test_splits(processed_images):
    """Create train/val/test splits, preventing business_id overlap."""
    business_to_photos = defaultdict(list)
    for photo_id, info in processed_images.items():
        business_to_photos[info['business_id']].append((photo_id, info['label']))
    
    business_ids = list(business_to_photos.keys())
    labels = [Counter(photo[1] for photo in photos).most_common(1)[0][0] 
              for photos in business_to_photos.values()]
    
    train_bids, temp_bids, _, _ = train_test_split(
        business_ids, labels, test_size=0.2, random_state=42, stratify=labels
    )
    val_bids, test_bids, _, _ = train_test_split(
        temp_bids, [labels[business_ids.index(bid)] for bid in temp_bids], 
        test_size=0.5, random_state=42, stratify=[labels[business_ids.index(bid)] for bid in temp_bids]
    )
    
    train_ids = [pid for bid in train_bids for pid, _ in business_to_photos[bid]]
    val_ids = [pid for bid in val_bids for pid, _ in business_to_photos[bid]]
    test_ids = [pid for bid in test_bids for pid, _ in business_to_photos[bid]]
    
    print(f"Train samples: {len(train_ids)}")
    print(f"Validation samples: {len(val_ids)}")
    print(f"Test samples: {len(test_ids)}")
    
    split_dirs = ['train', 'val', 'test']
    all_labels = set(info['label'] for info in processed_images.values())
    for split in split_dirs:
        for label in all_labels:
            os.makedirs(os.path.join(OUTPUT_DIR, split, label), exist_ok=True)
    
    def organize_files(id_list, split_name):
        moved_files = 0
        missing_files = 0
        for pid in tqdm(id_list, desc=f"Organizing {split_name} set"):
            info = processed_images[pid]
            src_path = info['path']
            if not os.path.exists(src_path):
                missing_files += 1
                continue
            dest_path = os.path.join(OUTPUT_DIR, split_name, info['label'], os.path.basename(src_path))
            shutil.move(src_path, dest_path)
            processed_images[pid]['path'] = dest_path
            moved_files += 1
        print(f"{split_name.capitalize()} set: Moved {moved_files}, Missing {missing_files}")
    
    organize_files(train_ids, 'train')
    organize_files(val_ids, 'val')
    organize_files(test_ids, 'test')
    
    split_info = {'train': train_ids, 'val': val_ids, 'test': test_ids}
    with open(os.path.join(OUTPUT_DIR, "split_info.json"), 'w') as f:
        json.dump(split_info, f, indent=2)
    
    return processed_images, split_info

# Run step
processed_images, split_info = create_train_val_test_splits(processed_images)

Train samples: 20815
Validation samples: 2473
Test samples: 2666


Organizing train set: 100%|██████████| 20815/20815 [00:49<00:00, 424.09it/s]


Train set: Moved 20815, Missing 0


Organizing val set: 100%|██████████| 2473/2473 [00:07<00:00, 335.27it/s]


Val set: Moved 2473, Missing 0


Organizing test set: 100%|██████████| 2666/2666 [00:08<00:00, 311.36it/s]


Test set: Moved 2666, Missing 0


### Step 7: Create Metadata Files

In [13]:
def create_metadata(processed_images, split_info):
    """Create metadata CSVs for train/val/test splits."""
    meta_dir = os.path.join(OUTPUT_DIR, "metadata")
    os.makedirs(meta_dir, exist_ok=True)
    
    def generate_metadata(id_list, split_name):
        metadata = []
        missing_files = 0
        for pid in tqdm(id_list, desc=f"Creating {split_name} metadata"):
            if pid not in processed_images:
                missing_files += 1
                continue
            info = processed_images[pid]
            file_path = info['path']
            if not os.path.exists(file_path):
                missing_files += 1
                continue
            rel_path = os.path.relpath(file_path, OUTPUT_DIR).replace("\\", "/")
            metadata.append({
                'photo_id': pid,
                'path': rel_path,
                'absolute_path': os.path.abspath(file_path),
                'label': info['label'],
                'business_id': info['business_id'],
                'caption': info['caption'],
                'split': split_name,
                'file_size': os.path.getsize(file_path),
                'modified_time': os.path.getmtime(file_path)
            })
        if missing_files:
            print(f"Warning: {missing_files} files missing from {split_name} set")
        return pd.DataFrame(metadata)
    
    train_meta = generate_metadata(split_info['train'], 'train')
    val_meta = generate_metadata(split_info['val'], 'val')
    test_meta = generate_metadata(split_info['test'], 'test')
    
    train_meta.to_csv(os.path.join(meta_dir, "train_metadata.csv"), index=False, encoding='utf-8')
    val_meta.to_csv(os.path.join(meta_dir, "val_metadata.csv"), index=False, encoding='utf-8')
    test_meta.to_csv(os.path.join(meta_dir, "test_metadata.csv"), index=False, encoding='utf-8')
    
    combined_meta = pd.concat([train_meta, val_meta, test_meta])
    combined_meta.to_csv(os.path.join(meta_dir, "combined_metadata.csv"), index=False, encoding='utf-8')
    
    stats = {
        'total_samples': len(combined_meta),
        'train_samples': len(train_meta),
        'val_samples': len(val_meta),
        'test_samples': len(test_meta),
        'class_distribution': combined_meta['label'].value_counts().to_dict()
    }
    with open(os.path.join(meta_dir, "dataset_stats.json"), 'w') as f:
        json.dump(stats, f, indent=2)
    
    print("Metadata creation complete!")
    print(f"Files saved in: {meta_dir}")

# Run step
create_metadata(processed_images, split_info)

Creating test metadata: 100%|██████████| 2666/2666 [00:01<00:00, 2395.60it/s]


Metadata creation complete!
Files saved in: C:\Users\Meet\Desktop\Yelp Photos\final_processed_data\metadata
